<a href="https://colab.research.google.com/github/Tengoku1/CS109_Causality_Counterfactual_Model/blob/main/CS109_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

import plotly.express as px

In [2]:
events = {
    "Self-driving car crash": {
        "factors": {
            "Human driver inattention": 0.8,
            "AI detection failure": 0.9,
            "Pedestrian mistake": 0.4,
            "Bad weather": 0.3
        },
        "weights": {
            "Human driver inattention": 0.30,
            "AI detection failure": 0.45,
            "Pedestrian mistake": 0.15,
            "Bad weather": 0.10
        },
        "public_blame": {
            "Human driver inattention": 65,
            "AI detection failure": 90,
            "Pedestrian mistake": 35,
            "Bad weather": 15
        }
    },

    "Medical treatment failure": {
        "factors": {
            "Doctor chose risky treatment": 0.7,
            "Patient was already high-risk": 0.9,
            "Drug side effect": 0.6,
            "Hospital delay": 0.4
        },
        "weights": {
            "Doctor chose risky treatment": 0.35,
            "Patient was already high-risk": 0.25,
            "Drug side effect": 0.25,
            "Hospital delay": 0.15
        },
        "public_blame": {
            "Doctor chose risky treatment": 80,
            "Patient was already high-risk": 25,
            "Drug side effect": 50,
            "Hospital delay": 60
        }
    },

    "Team loses championship game": {
        "factors": {
            "Missed final shot": 0.9,
            "Poor defense earlier": 0.8,
            "Coach strategy mistake": 0.6,
            "Referee mistake": 0.3
        },
        "weights": {
            "Missed final shot": 0.20,
            "Poor defense earlier": 0.45,
            "Coach strategy mistake": 0.25,
            "Referee mistake": 0.10
        },
        "public_blame": {
            "Missed final shot": 85,
            "Poor defense earlier": 45,
            "Coach strategy mistake": 55,
            "Referee mistake": 35
        }
    }
}

In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def compute_outcome_probability(factors, weights, bias=-1.5):
    """
    factors: dictionary of causal factor values
    weights: dictionary of causal weights
    bias: baseline difficulty of the outcome happening

    returns probability of bad outcome
    """

    x = np.array(list(factors.values()))
    w = np.array([weights[factor] for factor in factors.keys()])

    risk_score = np.dot(w, x) + bias

    return sigmoid(risk_score)

In [4]:
def compute_counterfactual_responsibility(event_name, event_data):
    factors = event_data["factors"]
    weights = event_data["weights"]
    public_blame = event_data["public_blame"]

    actual_probability = compute_outcome_probability(factors, weights)

    results = []

    for factor in factors:
        counterfactual_factors = factors.copy()

        # Remove that factor from the event
        counterfactual_factors[factor] = 0

        counterfactual_probability = compute_outcome_probability(
            counterfactual_factors,
            weights
        )

        responsibility = actual_probability - counterfactual_probability

        results.append({
            "Event": event_name,
            "Factor": factor,
            "Actual probability": actual_probability,
            "Counterfactual probability": counterfactual_probability,
            "AI responsibility score": responsibility * 100,
            "Public blame score": public_blame[factor]
        })

    return results

In [5]:
all_results = []

for event_name, event_data in events.items():
    event_results = compute_counterfactual_responsibility(event_name, event_data)
    all_results.extend(event_results)

df = pd.DataFrame(all_results)

df

,Event,Factor,Actual probability,Counterfactual probability,AI responsibility score,Public blame score
0,Self-driving car crash,Human driver inattention,0.317562,0.267959,4.960220,65
1,Self-driving car crash,AI detection failure,0.317562,0.236855,8.070671,90
2,Self-driving car crash,Pedestrian mistake,0.317562,0.304703,1.285837,35
3,Self-driving car crash,Bad weather,0.317562,0.311096,0.646562,15
4,Medical treatment failure,Doctor chose risky treatment,0.305764,0.256355,4.940855,80
5,Medical treatment failure,Patient was already high-risk,0.305764,0.260186,4.557727,25
6,Medical treatment failure,Drug side effect,0.305764,0.274881,3.088316,50
7,Medical treatment failure,Hospital delay,0.305764,0.293178,1.258588,60
8,Team loses championship game,Missed final shot,0.314320,0.276878,3.744169,85
9,Team loses championship game,Poor defense earlier,0.314320,0.242320,7.199953,45


In [6]:
print(df)

                           Event                         Factor  \
0         Self-driving car crash       Human driver inattention   
1         Self-driving car crash           AI detection failure   
2         Self-driving car crash             Pedestrian mistake   
3         Self-driving car crash                    Bad weather   
4      Medical treatment failure   Doctor chose risky treatment   
5      Medical treatment failure  Patient was already high-risk   
6      Medical treatment failure               Drug side effect   
7      Medical treatment failure                 Hospital delay   
8   Team loses championship game              Missed final shot   
9   Team loses championship game           Poor defense earlier   
10  Team loses championship game         Coach strategy mistake   
11  Team loses championship game                Referee mistake   

    Actual probability  Counterfactual probability  AI responsibility score  \
0             0.317562                    0.26795

In [7]:
fig = px.bar(
    df,
    x="Factor",
    y=["AI responsibility score", "Public blame score"],
    color_discrete_sequence=px.colors.qualitative.Set2,
    barmode="group",
    facet_col="Event",
    title="AI Counterfactual Responsibility vs Public Blame",
    labels={
        "value": "Score",
        "variable": "Judgment type",
        "Factor": "Causal factor"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=600
)

fig.show()

In [8]:
fig = px.scatter(
    df,
    x="AI responsibility score",
    y="Public blame score",
    color="Event",
    hover_name="Factor",
    title="Relationship Between AI Responsibility and Public Blame",
    labels={
        "AI responsibility score": "AI counterfactual responsibility score",
        "Public blame score": "Public blame score"
    },
    trendline="ols"
)

fig.show()

In [9]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


similarities = []

for event in df["Event"].unique():
    event_df = df[df["Event"] == event]

    ai_vector = event_df["AI responsibility score"].values
    public_vector = event_df["Public blame score"].values

    similarity = cosine_similarity(ai_vector, public_vector)

    similarities.append({
        "Event": event,
        "Cosine similarity": similarity
    })

similarity_df = pd.DataFrame(similarities)

similarity_df

,Event,Cosine similarity
0,Self-driving car crash,0.981229
1,Medical treatment failure,0.859803
2,Team loses championship game,0.826054


In [10]:
fig = px.bar(
    similarity_df,
    x="Event",
    y="Cosine similarity",
    title="Similarity Between AI Responsibility and Public Judgment",
    labels={
        "Cosine similarity": "Cosine similarity score",
        "Event": "Event"
    },
    range_y=[0, 1]
)

fig.show()

In [11]:
df = pd.read_csv("/content/Motor_Vehicle_Collisions_-_Crashes_20260602.csv", low_memory=False)

In [12]:
missing_percent = df.isna().mean().sort_values(ascending=False)
print(missing_percent)
cols_to_drop = missing_percent[missing_percent > 0.5].index
df = df.drop(columns=cols_to_drop)

VEHICLE TYPE CODE 5              0.995599
CONTRIBUTING FACTOR VEHICLE 5    0.995456
VEHICLE TYPE CODE 4              0.984029
CONTRIBUTING FACTOR VEHICLE 4    0.983420
VEHICLE TYPE CODE 3              0.930444
CONTRIBUTING FACTOR VEHICLE 3    0.927564
OFF STREET NAME                  0.820912
CROSS STREET NAME                0.383389
ZIP CODE                         0.304882
BOROUGH                          0.304755
ON STREET NAME                   0.219849
VEHICLE TYPE CODE 2              0.203936
CONTRIBUTING FACTOR VEHICLE 2    0.162427
LOCATION                         0.106251
LATITUDE                         0.106251
LONGITUDE                        0.106251
VEHICLE TYPE CODE 1              0.007519
CONTRIBUTING FACTOR VEHICLE 1    0.003619
NUMBER OF PERSONS KILLED         0.002624
NUMBER OF PERSONS INJURED        0.000008
CRASH DATE                       0.000000
NUMBER OF PEDESTRIANS INJURED    0.000000
CRASH TIME                       0.000000
NUMBER OF MOTORIST KILLED        0

In [ ]:
df

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,NUMBER OF PERSONS INJURED,...,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2
0,09/11/2021,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,2.0,...,0,0,0,2,0,Aggressive Driving/Road Rage,Unspecified,4455765,Sedan,Sedan
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,1.0,...,0,0,0,1,0,Pavement Slippery,NaN,4513547,Sedan,NaN
2,11/01/2023,1:29,BROOKLYN,11230.0,40.621790,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,1.0,...,0,0,0,1,0,Unspecified,Unspecified,4675373,Moped,Sedan
3,06/29/2022,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,0.0,...,0,0,0,0,0,Following Too Closely,Unspecified,4541903,Sedan,Pick-up Truck
4,09/21/2022,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,0.0,...,0,0,0,0,0,Passing Too Closely,Unspecified,4566131,Station Wagon/Sport Utility Vehicle,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2266215,05/29/2019,18:26,BRONX,10457.0,40.843280,-73.904630,"(40.84328, -73.90463)",ANTHONY AVENUE,EAST 173 STREET,2.0,...,0,0,0,0,0,Backing Unsafely,NaN,4141675,Station Wagon/Sport Utility Vehicle,NaN
2266216,05/31/2019,14:08,BROOKLYN,11226.0,40.655422,-73.959816,"(40.655422, -73.959816)",NaN,NaN,1.0,...,0,0,0,0,0,Unspecified,NaN,4146617,Sedan,NaN
2266217,06/10/2019,11:54,NaN,NaN,NaN,NaN,NaN,MAJOR DEEGAN EXPRESSWAY,WEST 161 STREET,0.0,...,0,0,0,0,0,Driver Inattention/Distraction,Unspecified,4149140,Sedan,Sedan
2266218,06/17/2019,16:51,NaN,NaN,40.817820,-73.915276,"(40.81782, -73.915276)",3 AVENUE,NaN,0.0,...,0,0,0,0,0,Unspecified,Unspecified,4153609,Sedan,Station Wagon/Sport Utility Vehicle


In [ ]:
# 1. Check missingness
missing_percent = df.isna().mean().sort_values(ascending=False)
print(missing_percent)

# 2. Drop columns with too much missing data
cols_to_drop = missing_percent[missing_percent > 0.5].index
df = df.drop(columns=cols_to_drop)

# 3. Drop rows missing the outcome variables
df = df.dropna(subset=[
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED"
])

# 4. Fill missing categorical predictors with Unknown
categorical_cols = [
    "CONTRIBUTING FACTOR VEHICLE 1",
    "CONTRIBUTING FACTOR VEHICLE 2",
    "VEHICLE TYPE CODE 1",
    "VEHICLE TYPE CODE 2",
    "BOROUGH"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

CROSS STREET NAME                0.383389
ZIP CODE                         0.304882
BOROUGH                          0.304755
ON STREET NAME                   0.219849
VEHICLE TYPE CODE 2              0.203936
CONTRIBUTING FACTOR VEHICLE 2    0.162427
LOCATION                         0.106251
LATITUDE                         0.106251
LONGITUDE                        0.106251
VEHICLE TYPE CODE 1              0.007519
CONTRIBUTING FACTOR VEHICLE 1    0.003619
NUMBER OF PERSONS KILLED         0.002624
NUMBER OF PERSONS INJURED        0.000008
CRASH TIME                       0.000000
CRASH DATE                       0.000000
NUMBER OF CYCLIST INJURED        0.000000
NUMBER OF PEDESTRIANS INJURED    0.000000
NUMBER OF PEDESTRIANS KILLED     0.000000
NUMBER OF MOTORIST KILLED        0.000000
NUMBER OF MOTORIST INJURED       0.000000
NUMBER OF CYCLIST KILLED         0.000000
COLLISION_ID                     0.000000
dtype: float64
